# Tutorial 6: Advanced Crystal Conditioning

**Time**: 30 minutes

**Topics**:
- Combined conditioning strategies
- Space group constraints
- Density targeting
- Polymorph generation

In [ ]:
import sys; sys.path.insert(0, '..')
import torch
import numpy as np
from crystal.conditioning import MolecularConditioning, SpaceGroupEmbedding, DensityConditioning

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Advanced Crystal Conditioning')

## 1. Conditioning Hierarchy

**Priority**:
1. Molecular features (REQUIRED)
2. Space group (OPTIONAL)
3. Density (OPTIONAL)

In [ ]:
# Setup all conditioning modules
mol_cond = MolecularConditioning(128, 256).to(device)
sg_emb = SpaceGroupEmbedding(64).to(device)
dens_cond = DensityConditioning(64).to(device)

print('Conditioning modules created:')
print(f'  Molecular: 256 dims')
print(f'  Space group: 64 dims')
print(f'  Density: 64 dims')
print(f'  Total: 384 dims')

## 2. Space Group Guide

**Common Crystal Systems**:
- Triclinic: 1-2
- Monoclinic: 3-15 (e.g., P2₁/c = 14)
- Orthorhombic: 16-74
- Tetragonal: 75-142
- Trigonal: 143-167
- Hexagonal: 168-194
- Cubic: 195-230 (e.g., Fm-3m = 225)

In [ ]:
# Example space groups
space_groups = {
    'P1': 1,
    'P2_1/c': 14,  # Most common
    'Pbca': 61,
    'P2_1/n': 14,
    'Fm-3m': 225,
}

for name, sg in space_groups.items():
    embedding = sg_emb(torch.tensor([sg], device=device))
    print(f'{name:10s} (#{sg:3d}): {embedding.shape}')

## 3. Density Targeting

In [ ]:
# Typical densities (g/cm³)
densities = {
    'Organic (light)': 0.8,
    'Typical organic': 1.2,
    'Organic (dense)': 1.6,
    'Inorganic (light)': 2.0,
    'Inorganic (heavy)': 3.0,
}

for name, dens in densities.items():
    context = dens_cond(torch.tensor([dens], device=device))
    print(f'{name:20s}: {dens:.1f} g/cm³ -> {context.shape}')

## 4. Combined Conditioning Strategy

In [ ]:
def prepare_full_conditioning(mol_features, space_group=None, density=None):
    contexts = []
    
    # 1. Molecular (REQUIRED)
    mol_context = mol_cond(mol_features)
    contexts.append(mol_context)
    print(f'Added molecular context: {mol_context.shape}')
    
    # 2. Space group (OPTIONAL)
    if space_group is not None:
        sg_context = sg_emb(torch.tensor([space_group], device=device))
        contexts.append(sg_context)
        print(f'Added space group context: {sg_context.shape}')
    
    # 3. Density (OPTIONAL)
    if density is not None:
        dens_context = dens_cond(torch.tensor([density], device=device))
        contexts.append(dens_context)
        print(f'Added density context: {dens_context.shape}')
    
    # Combine
    combined = torch.cat(contexts, dim=-1)
    print(f'\nCombined context: {combined.shape}')
    return combined

# Example: Full conditioning
# mol_features = extract_from_molecule(...)
# context = prepare_full_conditioning(mol_features, space_group=14, density=1.2)

## 5. Polymorph Generation

**Strategy**: Same molecule, different space groups

In [ ]:
# Generate polymorphs
polymorph_space_groups = [1, 14, 61, 225]  # Different symmetries

print('Polymorph generation strategy:')
for i, sg in enumerate(polymorph_space_groups, 1):
    print(f'\nPolymorph {i}:')
    print(f'  Space group: {sg}')
    print(f'  Same molecule features')
    print(f'  Different crystal structure')
    # context = prepare_full_conditioning(mol_features, space_group=sg)
    # crystal = model.sample(context=context)

## 6. Training Strategy

```bash
# Stage 1: Molecular only
python main_crystal.py \\
    --condition_on_molecule True \\
    --condition_on_space_group False \\
    --n_epochs 100

# Stage 2: Add space group
python main_crystal.py \\
    --resume outputs/stage1/model.pt \\
    --condition_on_molecule True \\
    --condition_on_space_group True \\
    --n_epochs 100

# Stage 3: Add density
python main_crystal.py \\
    --resume outputs/stage2/model.pt \\
    --condition_on_molecule True \\
    --condition_on_space_group True \\
    --condition_on_density True \\
    --n_epochs 100
```

## 7. No Fallback Validation

In [ ]:
def validate_conditioning_strict(space_group=None, density=None):
    errors = []
    
    # Space group validation
    if space_group is not None:
        if not isinstance(space_group, int):
            errors.append('Space group must be integer')
        elif space_group < 1 or space_group > 230:
            errors.append(f'Space group {space_group} not in [1, 230]')
    
    # Density validation
    if density is not None:
        if not isinstance(density, (int, float)):
            errors.append('Density must be numeric')
        elif density <= 0:
            errors.append('Density must be positive')
        elif density < 0.5 or density > 5.0:
            errors.append(f'Density {density} outside typical range [0.5, 5.0]')
    
    if errors:
        raise ValueError('Conditioning validation failed:\n' + '\n'.join(errors))
    
    return True

# Test
try:
    validate_conditioning_strict(space_group=14, density=1.2)
    print('✓ Valid conditioning')
except ValueError as e:
    print(f'✗ {e}')

try:
    validate_conditioning_strict(space_group=300)  # Invalid!
except ValueError as e:
    print(f'\n✓ Caught invalid input:\n{e}')

## Summary

### Learned:
1. ✅ Conditioning hierarchy
2. ✅ Space group selection
3. ✅ Density targeting
4. ✅ Combined strategies
5. ✅ Polymorph generation
6. ✅ Strict validation (no fallbacks)

### Best Practices:
- Always include molecular conditioning
- Use space group for symmetry control
- Use density for packing control
- Validate all inputs strictly
- Train incrementally

### Complete Example:
```python
# 1. Extract molecular features
mol_features = mol_encoder(h, x, edge_index)

# 2. Prepare conditioning
context = prepare_full_conditioning(
    mol_features,
    space_group=14,  # P2_1/c
    density=1.2      # g/cm³
)

# 3. Generate crystal
crystal = model.sample(
    n_samples=1,
    n_nodes=100,
    context=context,
    pbc=[True, True, True]
)

# 4. Export
writer.write_cif(crystal, 'output.cif', space_group=14)
```